In [ ]:
# %pip install -U qbraid qiskit numpy==2.4 pandas statsmodels scikit-learn tqdm

In [ ]:
# %%
from pathlib import Path
import json

import numpy as np
import pandas as pd
import statsmodels.api as sm
from qbraid import QbraidProvider
from sklearn.metrics import mean_absolute_error, mean_squared_error
from QRC_model_qbraid import QRC_Model_QBraid

# RUN SETTING

RUN_TYPE = "estimate"
# "estimate" = calculate expected tasks only; submits nothing
# "smoke"    = submit one circuit to verify the device works
# "subset"   = train and test the small QRC subset remotely

ALLOW_QPU_SUBMISSION = False
# False = safety lock; no QPU jobs may be submitted
# True  = allow "smoke" or "subset" to submit jobs
#
# Recommended workflow:
# 1. Start with RUN_TYPE = "estimate"
# 2. Check the task count and device
# 3. Set RUN_TYPE = "smoke" or "subset"
# 4. Set ALLOW_QPU_SUBMISSION = True

In [ ]:

DEVICE_ID = "aws:rigetti:qpu:cepheus-1-108q"
DATASET_PATH = Path("final_qrc_dataset.csv")
TICKER = None

NUM_QUBITS = 5
N_TROTTER = 2
DT = 0.1
FEEDBACKS = (0.1,)
SHOTS = 500
SEED = 0
RIDGE_PARAM = 1.0e-6
N_WASHOUT = 2

TRAIN_STEPS = 8
TEST_STEPS = 4

CREDIT_BUDGET = 900.0
PER_TASK_CREDIT = 30.0
PER_SHOT_CREDIT = 0.043

PRINT_JOB_IDS = True
OUTPUT_DIR = Path("qbraid_results")


In [ ]:
# %%
def print_device_metadata(device):
    print(f"Selected qBraid device: {device}")
    metadata_method = getattr(device, "metadata", None)

    if callable(metadata_method):
        try:
            print("Device metadata:")
            print(metadata_method())
        except Exception as exc:
            print(f"Could not retrieve device metadata: {exc}")


def estimate_and_print_cost(
    tasks,
    shots,
    per_task_credit,
    per_shot_credit,
    credit_budget,
):
    credits_per_task = per_task_credit + shots * per_shot_credit
    estimated_total = tasks * credits_per_task

    print("Remote execution estimate")
    print("-------------------------")
    print(f"Tasks:                {tasks}")
    print(f"Shots per task:       {shots}")
    print(f"Credits per task:     {credits_per_task:,.3f}")
    print(f"Estimated credits:    {estimated_total:,.3f}")
    print(f"Configured budget:    {credit_budget:,.3f}")
    print(f"Estimated remaining:  {credit_budget - estimated_total:,.3f}")

    return estimated_total

# %%
def build_har_dataframe(csv_path):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset not found: {csv_path.resolve()}")

    data = pd.read_csv(csv_path)

    required_columns = {"ticker", "date", "y", "split"}
    missing = required_columns.difference(data.columns)

    if missing:
        raise ValueError(
            "Dataset is missing required columns: "
            + ", ".join(sorted(missing))
        )

    data["date"] = pd.to_datetime(data["date"], errors="raise")
    data = data.sort_values(["ticker", "date"]).copy()

    grouped_y = data.groupby("ticker")["y"]

    data["y_d"] = grouped_y.shift(1)
    data["y_w"] = data.groupby("ticker")["y"].transform(
        lambda s: s.rolling(window=5).mean().shift(1)
    )
    data["y_m"] = data.groupby("ticker")["y"].transform(
        lambda s: s.rolling(window=22).mean().shift(1)
    )
    data["y_next"] = grouped_y.shift(-1)

    data["_next_split"] = data.groupby("ticker")["split"].shift(-1)
    data = data[data["_next_split"] == data["split"]].copy()

    data = data.dropna(
        subset=["y_d", "y_w", "y_m", "y_next", "split"]
    ).copy()

    train = data[data["split"] == "train"].copy()

    if train.empty:
        raise ValueError("No training rows remain after preprocessing.")

    X_train = sm.add_constant(
        train[["y_d", "y_w", "y_m"]],
        has_constant="add",
    )
    har_model = sm.OLS(train["y_next"], X_train).fit()

    X_all = sm.add_constant(
        data[["y_d", "y_w", "y_m"]],
        has_constant="add",
    )
    data["y_hat_HAR"] = har_model.predict(X_all)

    return data, har_model


def choose_ticker(data, requested=None):
    train_tickers = set(
        data.loc[data["split"] == "train", "ticker"].astype(str)
    )
    test_tickers = set(
        data.loc[data["split"] == "test", "ticker"].astype(str)
    )

    common = sorted(train_tickers.intersection(test_tickers))

    if not common:
        raise ValueError("No ticker is present in both splits.")

    if requested is not None:
        requested = str(requested)

        if requested not in common:
            raise ValueError(
                f"Ticker {requested!r} is unavailable. "
                f"Available choices: {common}"
            )

        return requested

    return common[0]


def build_single_ticker_arrays(
    data,
    ticker,
    train_steps,
    test_steps,
):
    ticker_data = data[
        data["ticker"].astype(str) == str(ticker)
    ].copy()

    train = (
        ticker_data[ticker_data["split"] == "train"]
        .sort_values("date")
        .tail(train_steps)
    )

    test = (
        ticker_data[ticker_data["split"] == "test"]
        .sort_values("date")
        .head(test_steps)
    )

    if len(train) < train_steps:
        raise ValueError(
            f"Ticker {ticker!r} has only {len(train)} training rows."
        )

    if len(test) < test_steps:
        raise ValueError(
            f"Ticker {ticker!r} has only {len(test)} testing rows."
        )

    x_train = train["y_next"].to_numpy(dtype=float)[None, :]
    yhar_train = train["y_hat_HAR"].to_numpy(dtype=float)[None, :]

    x_test = test["y_next"].to_numpy(dtype=float)[None, :]
    yhar_test = test["y_hat_HAR"].to_numpy(dtype=float)[None, :]
    test_dates = test["date"].to_numpy()

    return x_train, yhar_train, x_test, yhar_test, test_dates

# %%
def evaluate(y_true_log, y_pred_log, label):
    mse = mean_squared_error(y_true_log, y_pred_log)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true_log, y_pred_log)

    rv_true = np.exp(y_true_log)
    rv_pred = np.exp(y_pred_log)
    ratio = rv_true / rv_pred
    qlike = float(np.mean(ratio - np.log(ratio) - 1.0))

    metrics = {
        "mse": float(mse),
        "rmse": rmse,
        "mae": float(mae),
        "qlike": qlike,
    }

    print(
        f"{label:>12s}  "
        f"MSE: {metrics['mse']:.4f}  "
        f"RMSE: {metrics['rmse']:.4f}  "
        f"MAE: {metrics['mae']:.4f}  "
        f"QLIKE: {metrics['qlike']:.5f}"
    )

    return metrics


def create_qrc_model(device, max_tasks):
    return QRC_Model_QBraid(
        num_qubits=NUM_QUBITS,
        qbraid_device=device,
        ridge_param=RIDGE_PARAM,
        f_bs=FEEDBACKS,
        dt=DT,
        n_trotter=N_TROTTER,
        shots=SHOTS,
        seed=SEED,
        n_washout=N_WASHOUT,
        max_tasks=max_tasks,
        print_job_ids=PRINT_JOB_IDS,
    )

In [ ]:
# %%
# ============================================================
# CONNECT TO DEVICE
# ============================================================

provider = QbraidProvider()
device = provider.get_device(DEVICE_ID)
print_device_metadata(device)


# %% [markdown]
# ## Estimate task and credit usage
#
# RUN_TYPE options:
#
# - `"estimate"`: calculate cost only, no jobs submitted
# - `"smoke"`: submit one reservoir evolution
# - `"subset"`: train and test the bounded QRC subset
#
# QPU submission is permitted only when:
#
# ```python
# ALLOW_QPU_SUBMISSION = True
# ```


# %%
VALID_RUN_TYPES = {"estimate", "smoke", "subset"}

if RUN_TYPE not in VALID_RUN_TYPES:
    raise ValueError(
        f"RUN_TYPE must be one of {sorted(VALID_RUN_TYPES)}. "
        f"Received: {RUN_TYPE!r}"
    )


if RUN_TYPE == "smoke":
    REQUIRED_TASKS = len(FEEDBACKS)

else:
    # Both estimate and subset calculate the full subset requirement.
    REQUIRED_TASKS = QRC_Model_QBraid.estimate_required_tasks(
        train_steps=TRAIN_STEPS,
        test_steps=TEST_STEPS,
        n_reservoirs=len(FEEDBACKS),
        n_tickers=1,
        n_washout=N_WASHOUT,
    )


ESTIMATED_CREDITS = estimate_and_print_cost(
    tasks=REQUIRED_TASKS,
    shots=SHOTS,
    per_task_credit=PER_TASK_CREDIT,
    per_shot_credit=PER_SHOT_CREDIT,
    credit_budget=CREDIT_BUDGET,
)


if ESTIMATED_CREDITS > CREDIT_BUDGET:
    raise RuntimeError(
        "Estimated cost exceeds CREDIT_BUDGET. "
        "Reduce TRAIN_STEPS, TEST_STEPS, SHOTS, "
        "or the number of FEEDBACKS."
    )


print()
print(f"Run type:              {RUN_TYPE}")
print(f"QPU submission allowed: {ALLOW_QPU_SUBMISSION}")


# %%
# ============================================================
# CREATE OUTPUT DIRECTORY AND SAVE CONFIGURATION
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

experiment_configuration = {
    "run_type": RUN_TYPE,
    "allow_qpu_submission": ALLOW_QPU_SUBMISSION,
    "device": DEVICE_ID,
    "dataset": str(DATASET_PATH),
    "requested_ticker": TICKER,
    "num_qubits": NUM_QUBITS,
    "n_trotter": N_TROTTER,
    "dt": DT,
    "feedbacks": list(FEEDBACKS),
    "num_reservoirs": len(FEEDBACKS),
    "shots": SHOTS,
    "seed": SEED,
    "ridge_param": RIDGE_PARAM,
    "washout": N_WASHOUT,
    "train_steps": TRAIN_STEPS,
    "test_steps": TEST_STEPS,
    "required_tasks": REQUIRED_TASKS,
    "credit_budget": CREDIT_BUDGET,
    "per_task_credit": PER_TASK_CREDIT,
    "per_shot_credit": PER_SHOT_CREDIT,
    "estimated_credits": ESTIMATED_CREDITS,
}

configuration_path = OUTPUT_DIR / "qbraid_experiment_configuration.json"

configuration_path.write_text(
    json.dumps(
        experiment_configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Configuration saved: {configuration_path}")


# %% [markdown]
# ## Estimate-only mode
#
# This mode stops before loading data or submitting quantum jobs.


# %%
if RUN_TYPE == "estimate":
    print()
    print("Estimate completed.")
    print("No quantum jobs were submitted.")
    print(
        "Change RUN_TYPE to 'smoke' or 'subset' and set "
        "ALLOW_QPU_SUBMISSION = True when ready."
    )


# %% [markdown]
# ## Optional one-task smoke test
#
# This submits one `evolve_qrc()` call per reservoir.
#
# For one feedback value:
#
# ```python
# FEEDBACKS = (0.1,)
# ```
#
# the smoke test submits one task.


# %%
if RUN_TYPE != "smoke":
    print("Smoke test skipped.")

elif not ALLOW_QPU_SUBMISSION:
    print(
        "Smoke test is locked. "
        "Set ALLOW_QPU_SUBMISSION = True to submit the job."
    )

else:
    smoke_model = create_qrc_model(
        device=device,
        max_tasks=REQUIRED_TASKS,
    )

    smoke_features = smoke_model.evolve_qrc(t0=0.1)

    smoke_output = {
        "configuration": experiment_configuration,
        "feature_length": len(smoke_features),
        "features": np.asarray(
            smoke_features,
            dtype=float,
        ).tolist(),
        "job_ids": list(smoke_model.job_ids),
        "tasks_submitted": smoke_model.submitted_tasks,
        "measurement_records": getattr(
            smoke_model,
            "measurement_records",
            [],
        ),
    }

    smoke_path = OUTPUT_DIR / "qbraid_smoke_result.json"

    smoke_path.write_text(
        json.dumps(
            smoke_output,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("Smoke test completed.")
    print(f"Feature length: {len(smoke_features)}")
    print(f"Features:       {np.asarray(smoke_features)}")
    print(f"Job IDs:        {smoke_model.job_ids}")
    print(f"Saved result:   {smoke_path}")


# %% [markdown]
# ## Prepare bounded dataset subset
#
# This loads and prepares the data.
#
# It does not submit quantum jobs.


# %%
if RUN_TYPE == "subset":
    qrc_data, har_model = build_har_dataframe(DATASET_PATH)
    selected_ticker = choose_ticker(qrc_data, TICKER)

    (
        x_train,
        yhar_train,
        x_test,
        yhar_test,
        test_dates,
    ) = build_single_ticker_arrays(
        data=qrc_data,
        ticker=selected_ticker,
        train_steps=TRAIN_STEPS,
        test_steps=TEST_STEPS,
    )

    print()
    print(f"Selected ticker: {selected_ticker}")
    print(f"x_train shape:   {x_train.shape}")
    print(f"x_test shape:    {x_test.shape}")

else:
    print("Subset preparation skipped.")


# %% [markdown]
# ## Submit subset experiment
#
# Remote tasks are submitted only when:
#
# ```python
# RUN_TYPE == "subset"
# ALLOW_QPU_SUBMISSION is True
# ```
#
# Measurement records should be autosaved by `QRC_Model_QBraid`
# after each successful quantum task.


# %%
if RUN_TYPE != "subset":
    print("Subset execution skipped.")

elif not ALLOW_QPU_SUBMISSION:
    print(
        "Subset execution is locked. "
        "Set ALLOW_QPU_SUBMISSION = True after reviewing the estimate."
    )

else:
    qrc_model = create_qrc_model(
        device=device,
        max_tasks=REQUIRED_TASKS,
    )

    # --------------------------------------------------------
    # Quantum feature generation and Ridge fitting
    # --------------------------------------------------------

    qrc_model.train(
        x_train,
        yhar_train,
    )

    qrc_model.fit()

    qrc_predictions = qrc_model.forward_one_shot(
        x_test,
        yhar_test,
    )

    # forward_one_shot predicts x_test[:, 1:]
    y_true = x_test[0, 1:]
    y_har = yhar_test[0, 1:]
    y_qrc = qrc_predictions[0]
    aligned_dates = test_dates[1:]


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    har_metrics = evaluate(
        y_true,
        y_har,
        label="HAR only",
    )

    qrc_metrics = evaluate(
        y_true,
        y_qrc,
        label="HAR + QRC",
    )


    # --------------------------------------------------------
    # Output paths
    # --------------------------------------------------------

    safe_ticker = str(selected_ticker).replace("/", "_")

    predictions_path = (
        OUTPUT_DIR
        / f"qbraid_predictions_{safe_ticker}.csv"
    )

    metrics_path = (
        OUTPUT_DIR
        / f"qbraid_metrics_{safe_ticker}.json"
    )

    jobs_path = (
        OUTPUT_DIR
        / f"qbraid_job_ids_{safe_ticker}.json"
    )

    measurements_path = (
        OUTPUT_DIR
        / f"qbraid_measurements_{safe_ticker}.json"
    )

    model_state_path = (
        OUTPUT_DIR
        / f"qbraid_model_state_{safe_ticker}.json"
    )


    # --------------------------------------------------------
    # Save aligned predictions
    # --------------------------------------------------------

    results = pd.DataFrame(
        {
            "date": aligned_dates,
            "ticker": selected_ticker,
            "y_true": y_true,
            "y_har": y_har,
            "y_qrc": y_qrc,
        }
    )

    results.to_csv(
        predictions_path,
        index=False,
    )


    # --------------------------------------------------------
    # Save metrics and experiment information
    # --------------------------------------------------------

    metrics_output = {
        "configuration": {
            **experiment_configuration,
            "selected_ticker": selected_ticker,
        },
        "tasks_submitted": qrc_model.submitted_tasks,
        "har": har_metrics,
        "har_plus_qrc": qrc_metrics,
    }

    metrics_path.write_text(
        json.dumps(
            metrics_output,
            indent=2,
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Save job IDs
    # --------------------------------------------------------

    jobs_path.write_text(
        json.dumps(
            {
                "device": DEVICE_ID,
                "ticker": selected_ticker,
                "job_ids": list(qrc_model.job_ids),
            },
            indent=2,
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Save raw counts and Z expectation values
    # --------------------------------------------------------

    measurement_records = getattr(
        qrc_model,
        "measurement_records",
        [],
    )

    measurements_path.write_text(
        json.dumps(
            {
                "configuration": {
                    **experiment_configuration,
                    "selected_ticker": selected_ticker,
                },
                "records": measurement_records,
            },
            indent=2,
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Save QRC feature matrix and targets
    # --------------------------------------------------------

    train_features = np.asarray(
        qrc_model.train_features,
        dtype=float,
    )

    train_targets = np.asarray(
        qrc_model.train_outputs,
        dtype=float,
    )


    # --------------------------------------------------------
    # Try to save fitted Ridge parameters
    # --------------------------------------------------------

    ridge_state = {}

    ridge_object = getattr(
        qrc_model,
        "ridge",
        None,
    )

    if ridge_object is not None:
        if hasattr(ridge_object, "coef_"):
            ridge_state["coef"] = np.asarray(
                ridge_object.coef_,
                dtype=float,
            ).tolist()

        if hasattr(ridge_object, "intercept_"):
            ridge_state["intercept"] = np.asarray(
                ridge_object.intercept_,
                dtype=float,
            ).tolist()

        if hasattr(ridge_object, "alpha"):
            alpha_value = ridge_object.alpha

            if np.isscalar(alpha_value):
                ridge_state["alpha"] = float(alpha_value)
            else:
                ridge_state["alpha"] = np.asarray(
                    alpha_value,
                    dtype=float,
                ).tolist()


    # --------------------------------------------------------
    # Save complete reusable classical state
    # --------------------------------------------------------

    model_state = {
        "configuration": {
            **experiment_configuration,
            "selected_ticker": selected_ticker,
        },
        "train_features": train_features.tolist(),
        "train_targets": train_targets.tolist(),
        "x_train": np.asarray(
            x_train,
            dtype=float,
        ).tolist(),
        "yhar_train": np.asarray(
            yhar_train,
            dtype=float,
        ).tolist(),
        "x_test": np.asarray(
            x_test,
            dtype=float,
        ).tolist(),
        "yhar_test": np.asarray(
            yhar_test,
            dtype=float,
        ).tolist(),
        "test_dates": [
            str(date)
            for date in test_dates
        ],
        "predictions": {
            "y_true": np.asarray(
                y_true,
                dtype=float,
            ).tolist(),
            "y_har": np.asarray(
                y_har,
                dtype=float,
            ).tolist(),
            "y_qrc": np.asarray(
                y_qrc,
                dtype=float,
            ).tolist(),
        },
        "ridge": ridge_state,
        "job_ids": list(qrc_model.job_ids),
        "measurement_records": measurement_records,
        "metrics": {
            "har": har_metrics,
            "har_plus_qrc": qrc_metrics,
        },
    }

    model_state_path.write_text(
        json.dumps(
            model_state,
            indent=2,
        ),
        encoding="utf-8",
    )


    # --------------------------------------------------------
    # Completion summary
    # --------------------------------------------------------

    print()
    print("Subset run completed.")
    print(
        f"Tasks submitted: "
        f"{qrc_model.submitted_tasks}/{REQUIRED_TASKS}"
    )
    print(f"Predictions:      {predictions_path}")
    print(f"Metrics:          {metrics_path}")
    print(f"Job IDs:          {jobs_path}")
    print(f"Measurements:     {measurements_path}")
    print(f"Reusable state:   {model_state_path}")